# Target Definition and Validation

Objective: Understand the LateAircraftDelay column and define a clear delay propagation target variable based on data exploration.


In [1]:
import pandas as pd

In [2]:
df = pd.read_parquet('../data/processed/flight_data_clean.parquet')
df.head()

,Year,Quarter,Month,DayofMonth,DayOfWeek,FlightDate,Marketing_Airline_Network,Operated_or_Branded_Code_Share_Partners,DOT_ID_Marketing_Airline,IATA_Code_Marketing_Airline,...,Div4TailNum,Div5Airport,Div5AirportID,Div5AirportSeqID,Div5WheelsOn,Div5TotalGTime,Div5LongestGTime,Div5WheelsOff,Div5TailNum,Duplicate
0,2024,1,1,14,7,2024-01-14,UA,UA_CODESHARE,19977,UA,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N
1,2024,1,1,14,7,2024-01-14,UA,UA_CODESHARE,19977,UA,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N
2,2024,1,1,14,7,2024-01-14,UA,UA_CODESHARE,19977,UA,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N
3,2024,1,1,14,7,2024-01-14,UA,UA_CODESHARE,19977,UA,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N
4,2024,1,1,14,7,2024-01-14,UA,UA_CODESHARE,19977,UA,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N


In [3]:
df["LateAircraftDelay"].dtype

dtype('float64')

In [4]:
df["LateAircraftDelay"].head(10)

0    45.0
1     0.0
2     0.0
3     0.0
4     0.0
5     1.0
6     NaN
7     NaN
8     NaN
9     0.0
Name: LateAircraftDelay, dtype: float64

In [5]:
df["LateAircraftDelay"].describe()

count    134575.000000
mean         30.582961
std          68.247021
min           0.000000
25%           0.000000
50%           1.000000
75%          35.000000
max        1741.000000
Name: LateAircraftDelay, dtype: float64

In [6]:
(df["LateAircraftDelay"] > 0).value_counts()

LateAircraftDelay
False    490942
True      67773
Name: count, dtype: int64

In [7]:
df["LateAircraftDelay"].isnull().sum()

424140

In [8]:
df.loc[df["LateAircraftDelay"] > 0, 
       ["FlightDate", "Origin", "Dest", "DepDelay", "LateAircraftDelay"]].head(10)

,FlightDate,Origin,Dest,DepDelay,LateAircraftDelay
0,2024-01-14,MHT,EWR,71.0,45.0
5,2024-01-14,STL,ORD,52.0,1.0
20,2024-01-14,ORD,RIC,57.0,2.0
50,2024-01-14,SGF,DEN,225.0,214.0
59,2024-01-14,LIT,DEN,94.0,85.0
64,2024-01-14,SGF,DEN,12.0,5.0
66,2024-01-14,LFT,IAH,117.0,104.0
77,2024-01-14,PVD,IAD,17.0,6.0
78,2024-01-14,JAN,IAH,233.0,196.0
83,2024-01-14,IAH,LIT,53.0,50.0


In [9]:
propagation_delays = df.loc[df["LateAircraftDelay"] > 0, "LateAircraftDelay"]

In [10]:
propagation_delays.describe()

count    67773.000000
mean        60.727753
std         86.127787
min          1.000000
25%         17.000000
50%         35.000000
75%         73.000000
max       1741.000000
Name: LateAircraftDelay, dtype: float64

In [11]:
bins = [0, 5, 10, 15, 30, 60, 120, propagation_delays.max()]
labels = ["1-5", "6-10", "11-15", "16-30", "31-60", "61-120", "120+"]

delay_bins = pd.cut(propagation_delays, bins=bins, labels=labels, include_lowest=True)

delay_bins.value_counts(normalize=True).sort_index()

LateAircraftDelay
1-5       0.058047
6-10      0.066251
11-15     0.084385
16-30     0.244448
31-60     0.240140
61-120    0.182580
120+      0.124150
Name: proportion, dtype: float64

In [ ]:
# Define delay propagation target
df["delay_propagation"] = (df["LateAircraftDelay"] > 0).astype(int)

df["delay_propagation"].value_counts(normalize=True)

delay_propagation
0    0.878698
1    0.121302
Name: proportion, dtype: float64

In [13]:
# Drop leakage column (used only for target creation)
df = df.drop(columns=["LateAircraftDelay"])

In [14]:
df.to_parquet("../data/processed/flight_data_with_target.parquet", index=False)

### Summary

This notebook focused on defining a clear and meaningful target variable for delay propagation.

The `LateAircraftDelay` column was examined in detail to understand what it represents and how it behaves in the dataset. Its data type, missing values, and summary statistics were reviewed. The distribution of delay durations was also analyzed using grouped ranges (such as 1–5 minutes, 6–10 minutes, and higher intervals) to understand whether most cases were minor delays or operationally significant ones.

The analysis showed that only a small percentage of propagation cases were very short delays, while most cases involved meaningful delay durations. Based on this observation, delay propagation was defined as any case where `LateAircraftDelay` is greater than 0.

A new binary column, `delay_propagation`, was created to indicate whether delay spread occurred (1) or not (0). The original `LateAircraftDelay` column was then removed to avoid data leakage in later modeling stages.

At this point, the dataset contains a well-defined target variable and is ready for feature engineering and guided exploratory analysis in the next stage.